<a href="https://colab.research.google.com/github/lukmannm/data-science-2026/blob/main/Pertemuan10_Lukman_240401010181.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 10 — Customer Churn Prediction: Random Forest

*   Nama: Lukman Nur Hakim  
*   NIM: 240401010181
*   Dataset: Telco Customer Churn





## 1. Load Data

Dataset Telco Customer Churn berisi sekitar 7.043 pelanggan dengan 19 fitur.
Target: `Churn` (Yes = churn, No = tidak churn).
Proporsi churn sekitar 26,5% sehingga dataset bersifat imbalanced.


In [5]:
import pandas as pd

df = pd.read_csv("/content/sample_data/telco_churn.csv")
print("Shape:", df.shape)
print("\nDistribusi Churn:")
print(df["Churn"].value_counts(normalize=True).round(3))

Shape: (7043, 21)

Distribusi Churn:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


## 2. Preprocessing

In [6]:
from sklearn.model_selection import train_test_split

# Drop customerID (tidak relevan)
df = df.drop(columns=["customerID"])

# TotalCharges bertipe object, konversi ke numerik
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

# Encoding target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# One-Hot Encoding fitur kategorikal
df = pd.get_dummies(df, drop_first=True)

print("Shape setelah encoding:", df.shape)

# Pisahkan fitur dan target
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Train-Test Split stratified
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train: {X_tr.shape[0]} baris, Test: {X_te.shape[0]} baris")
print(f"Proporsi churn di train: {y_tr.mean():.3f}")
print(f"Proporsi churn di test : {y_te.mean():.3f}")

Shape setelah encoding: (7032, 31)
Train: 5625 baris, Test: 1407 baris
Proporsi churn di train: 0.266
Proporsi churn di test : 0.266


- Drop kolom customerID karena tidak relevan sebagai fitur
- Konversi TotalCharges dari object ke numerik (ada spasi kosong di data asli)
- Encoding target Churn: Yes=1, No=0
- One-Hot Encoding fitur kategorikal menggunakan pd.get_dummies(drop_first=True)
- Train-Test Split 80:20 dengan stratify=y agar proporsi kelas tetap seimbang di kedua set

## 3. Latih Model Random Forest

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42)

rf.fit(X_tr, y_tr)
print("Model selesai dilatih!")
print(f"Jumlah fitur: {rf.n_features_in_}")

Model selesai dilatih!
Jumlah fitur: 30


RandomForestClassifier dengan class_weight="balanced" digunakan untuk
menangani ketidakseimbangan kelas (imbalanced dataset). Parameter ini membuat
model memberi bobot lebih besar pada kelas minoritas (churn) sehingga model
tidak bias ke kelas mayoritas (tidak churn).

## 4. Evaluasi

In [8]:
from sklearn.metrics import classification_report, roc_auc_score

# Prediksi dan probabilitas
y_pred  = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]

# Classification report
print("=== Classification Report ===")
print(classification_report(y_te, y_pred, target_names=["No Churn", "Churn"]))

# ROC-AUC
roc_auc = roc_auc_score(y_te, y_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

    No Churn       0.83      0.90      0.86      1033
       Churn       0.63      0.49      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407

ROC-AUC Score: 0.8199



Metrik utama yang diperhatikan adalah Recall dan ROC-AUC untuk kelas churn (kelas 1).
Recall penting karena lebih berbahaya melewatkan pelanggan yang akan churn (False Negative)
daripada salah memprediksi pelanggan yang tidak churn sebagai churn (False Positive).

## 5. Probabilitas

In [9]:
# Buat dataframe probabilitas churn per pelanggan
churn_proba_df = pd.DataFrame({
    "Churn_Probability": y_proba,
    "Actual_Churn"     : y_te.values
}).reset_index(drop=True)

# Tampilkan 10 pelanggan dengan risiko churn tertinggi
print("=== 10 Pelanggan Risiko Churn Tertinggi ===")
print(churn_proba_df.sort_values("Churn_Probability", ascending=False).head(10))

# Ringkasan
print(f"\nRata-rata probabilitas churn : {y_proba.mean():.3f}")
print(f"Pelanggan berisiko tinggi (prob > 0.7): {(y_proba > 0.7).sum()} pelanggan")

=== 10 Pelanggan Risiko Churn Tertinggi ===
      Churn_Probability  Actual_Churn
369            1.000000             1
1220           0.996667             1
728            0.996667             0
107            0.996667             0
304            0.996667             1
31             0.990000             1
261            0.983333             1
591            0.970000             1
1149           0.963333             1
446            0.960000             0

Rata-rata probabilitas churn : 0.276
Pelanggan berisiko tinggi (prob > 0.7): 151 pelanggan


## Kesimpulan

Model Random Forest dengan class_weight="balanced" berhasil mendeteksi pelanggan
yang berpotensi churn dengan ROC-AUC sebesar 0.8199, yang menunjukkan kemampuan
diskriminasi model yang cukup baik. Model menunjukkan Recall kelas churn sebesar 0.49,
artinya model mampu mengidentifikasi 49% pelanggan yang benar-benar akan churn.
Dari 1.407 pelanggan dalam data test, terdapat 151 pelanggan dengan probabilitas
churn di atas 70% yang perlu diprioritaskan oleh tim retensi. Dengan menggunakan
model ini, perusahaan dapat mengalokasikan sumber daya secara lebih efisien untuk
mencegah kehilangan pelanggan sebelum churn benar-benar terjadi.